In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [4]:
simulator = AerSimulator()

In [5]:
# Helper Functions

def quantum_random_bit():
    """Generate one random bit by measuring |+> = (|0> + |1>) / sqrt(2)."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])


def quantum_random_bits(n):
    """Generate n quantum random bits without using Python's random module."""
    return [quantum_random_bit() for _ in range(n)]


def prepare_qubit(bit, basis):
    """
    Alice prepares one qubit.

    basis = 0: computational / rectilinear basis
        bit 0 -> |0>
        bit 1 -> |1>

    basis = 1: diagonal basis
        bit 0 -> |+>
        bit 1 -> |->
    """
    qc = QuantumCircuit(1, 1)

    if bit == 1:
        qc.x(0)

    if basis == 1:
        qc.h(0)

    return qc


def measure_qubit(qubit_circuit, basis):
    """
    Measure one qubit in the chosen basis.

    basis = 0: computational / rectilinear basis
    basis = 1: diagonal basis
    """
    qc = qubit_circuit.copy()

    if basis == 1:
        qc.h(0)

    qc.measure(0, 0)
    result = simulator.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])


def bit_string(bits):
    """Format a list of bits as a compact string."""
    return ''.join(str(bit) for bit in bits)

In [7]:
# Running the Protocol

def run_bb84_plain(num_qubits=100, sample_size=20, threshold=0.20):
    print("BB84 without attacker")
    print("=" * 50)

    # ========================================================
    # Alice
    # ========================================================
    alice_bits = quantum_random_bits(num_qubits)
    alice_bases = quantum_random_bits(num_qubits)
    qubits = [prepare_qubit(bit, basis) for bit, basis in zip(alice_bits, alice_bases)]

    print("Alice prepared", num_qubits, "qubits.")

    # ========================================================
    # Bob
    # ========================================================
    bob_bases = quantum_random_bits(num_qubits)
    bob_bits = [measure_qubit(qubit, basis) for qubit, basis in zip(qubits, bob_bases)]

    print("Bob measured the qubits.")

    # ========================================================
    # Public basis comparison and key sifting
    # ========================================================
    matching_indices = [i for i in range(num_qubits) if alice_bases[i] == bob_bases[i]]

    alice_sifted_key = [alice_bits[i] for i in matching_indices]
    bob_sifted_key = [bob_bits[i] for i in matching_indices]

    print("Number of matching bases:", len(matching_indices))
    print("Alice sifted key:", bit_string(alice_sifted_key))
    print("Bob sifted key:  ", bit_string(bob_sifted_key))

    # ========================================================
    # Test part of the sifted key for errors
    # ========================================================
    if len(alice_sifted_key) < sample_size:
        print("Not enough matching bits for the chosen sample size.")
        return [], []

    alice_test = alice_sifted_key[:sample_size]
    bob_test = bob_sifted_key[:sample_size]

    errors = sum(1 for a, b in zip(alice_test, bob_test) if a != b)
    error_rate = errors / sample_size

    print("Testing first", sample_size, "sifted bits.")
    print("Errors:", errors)
    print("Error rate:", round(error_rate, 3))
    print("Detection threshold:", threshold)

    if error_rate > threshold:
        print("Attack/noise detected. Key discarded.")
        return [], []

    final_alice_key = alice_sifted_key[sample_size:]
    final_bob_key = bob_sifted_key[sample_size:]

    print("No attack detected. Key accepted.")
    print("Final Alice key:", bit_string(final_alice_key))
    print("Final Bob key:  ", bit_string(final_bob_key))

    return final_alice_key, final_bob_key



In [8]:
# Expected Result

final_alice_key, final_bob_key = run_bb84_plain(
    num_qubits=100,
    sample_size=20,
    threshold=0.20
)

BB84 without attacker
Alice prepared 100 qubits.
Bob measured the qubits.
Number of matching bases: 59
Alice sifted key: 10111101101000011100010010010000010010101000010010011101110
Bob sifted key:   10111101101000011100010010010000010010101000010010011101110
Testing first 20 sifted bits.
Errors: 0
Error rate: 0.0
Detection threshold: 0.2
No attack detected. Key accepted.
Final Alice key: 010010010000010010101000010010011101110
Final Bob key:   010010010000010010101000010010011101110
